In [ ]:
import sqlite3
import pandas as pd

from IPython.display import display
from pathlib import Path
from collections import defaultdict

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [ ]:
PROJECT_PATH = Path.cwd().parent
DATA_PATH = PROJECT_PATH / "data"
PARQUETS_PATH = PROJECT_PATH / "data" / "processed"

In [ ]:
PARQUETS = list(PARQUETS_PATH.glob("*.parquet"))

In [ ]:
# Create a dictionary with all columns from all datasets
# Dict value is a set, in case a columns has a type missmatch between datasets
# We use defaultdict to avoid having if statements

COLUMNS = defaultdict(set)
DFS = list()

for p in PARQUETS:
    df = pd.read_parquet(p)
    DFS.append(df)
    for c in sorted(df.columns):
        COLUMNS[c].add(df[c].dtype)

print("Column names and their type(s): ", end="\n\n")
for k,v in COLUMNS.items():
    print(f"{k}: {v}")

for k in COLUMNS.keys():
    if len(COLUMNS[k]) > 1:
        print(f"Column {k} has more than one type.")

In [ ]:
# See what columns each dataset is missing

for p in PARQUETS:
    df = pd.read_parquet(p)
    missing = sorted(COLUMNS.keys() - sorted(df.columns))
    print(f"Parquet {p.name} misses columns: {missing}")

In [ ]:
# Create a dictionary containing fixes for each dataset
# Then iterate over them and apply the changes

fixes = {
    "eurobank_2021-06_2026-05.parquet": {
        "amount_euro": lambda df : df["amount"],
        "currency": "EUR",
        "fee": 0.0
    },
    "lunar_2024-10_2026-05.parquet": {"fee": 0.0},
    "revolut_euro_2020-02_2026-05.parquet": {"amount_euro": lambda df : df["amount"]}
}

for p in PARQUETS:
    df = pd.read_parquet(p)
    if p.name in fixes:
        parquet_fix = fixes[p.name]
        for new_col, val in parquet_fix.items():
            df[new_col] = val(df) if callable(val) else val
        # Save new parquet
        df.to_parquet(PARQUETS_PATH / p.name)
        
    # print(sorted(df.columns))
    # display(df.head(2))

In [ ]:
# Create Database

def create_db() -> sqlite3.Connection:
    """
    Create database and populate transactions table with data from the processed parquets.
    """
    db_path = DATA_PATH / "transactions.db"
    
    if db_path.exists():
        return sqlite3.connect(db_path)
    
    conn = sqlite3.connect(db_path)
    for p in PARQUETS:
        df = pd.read_parquet(p)
        df.to_sql("transactions", conn, if_exists="append", index=False) # if_exists refers to transactions table
        print(f"Loaded {p.name}: {len(df)} rows")
    
    conn.execute("CREATE INDEX IF NOT EXISTS idx_date ON transactions(date)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_category ON transactions(category)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_source ON transactions(source)")
    
    return conn

conn = create_db()